In [3]:
import pandas as pd
import ast

In [5]:
current_db = pd.read_csv("./data/umls/mrconso_filtered_db_and_sty_474316_drug_chemical_level_0_9.csv")

In [7]:
current_db.shape

(474316, 5)

In [9]:
current_db.head()

,cui,srl,sab,str,sty
0,C5687455,9,SNOMEDCT_US,Solution for infusion and/or injection,Biomedical or Dental Material
1,C5687658,9,SNOMEDCT_US,Amlodipine- and atenolol-containing product in...,Clinical Drug
2,C5688824,9,SNOMEDCT_US,Prolonged-release periodontal insert,Biomedical or Dental Material
3,C5690295,0,MSH,ACEA100610,Organic Chemical
4,C5690315,0,MSH,"MIRN3113 microRNA, human",Biologically Active Substance


In [11]:
current_db.cui.nunique()

474316

In [13]:
full_db = pd.read_csv("./data/umls/mrconso_filtered_db_and_sty_20251116.csv")[['cui','ispref','sab','str','scui','sty']]
full_db = full_db.drop_duplicates(subset=['str', 'scui'], keep='first')
full_db['cui']=full_db['cui'].str.strip()

In [49]:
full_db.cui.nunique()

902483

In [53]:
full_db

,cui,ispref,sab,str,scui,sty
0,C5686927,Y,SNOMEDCT_US,Cholesterol atheroembolism of cerebrum,1204128009,Disease or Syndrome
1,C5686927,Y,SNOMEDCT_US,Cholesterol atheroembolism to cerebrum,1204128009,Disease or Syndrome
2,C5686927,Y,SNOMEDCT_US,Cholesterol atheromatous embolism of cerebrum,1204128009,Disease or Syndrome
3,C5686927,Y,SNOMEDCT_US,Cholesterol atheromatous embolism of cerebrum ...,1204128009,Disease or Syndrome
4,C5686991,Y,SNOMEDCT_US,Bilateral contracture of muscle of thighs,15703881000119104,Anatomical Abnormality
...,...,...,...,...,...,...
2566148,C5686520,Y,SNOMEDCT_US,Carboprost (as carboprost tromethamine) 125 mi...,1201833005,Clinical Drug
2566149,C5686520,Y,SNOMEDCT_US,Product containing precisely carboprost (as ca...,1201833005,Clinical Drug
2566150,C5686520,Y,SNOMEDCT_US,Carboprost (as carboprost trometamol) 125 micr...,1201833005,Clinical Drug
2566151,C5686774,Y,SNOMEDCT_US,Local flap reconstruction by W-plasty - action,1204229009,Therapeutic or Preventive Procedure


In [14]:
drugbank = pd.read_csv("./data/term_dictionaries/drugbank_full_20251115/drugbank_full_database_synonyms.csv")
drugbank.shape

(17430, 8)

In [15]:
drugbank.head()

,primary_drugbank_id,all_drugbank_ids,primary_name,synonyms,brand_names,product_names,all_names,external_identifiers
0,DB00001,['DB00001'],Lepirudin,[],[],[],['Lepirudin'],NaN
1,DB01022,['DB01022'],Phylloquinone,[],[],[],['Phylloquinone'],NaN
2,DB01373,['DB01373'],Calcium,[],[],[],['Calcium'],NaN
3,DB00002,['DB00002'],Cetuximab,[],[],[],['Cetuximab'],NaN
4,DB00003,"['DB00003', 'BTD00001', 'BIOD00001']",Dornase alfa,['Deoxyribonuclease (human clone 18-1 protein ...,['Viscozyme'],"['Pulmozyme', 'Pulmozyme']",['Deoxyribonuclease (human clone 18-1 protein ...,Drugs Product Database (DPD):650;PubChem Subst...


In [17]:
ast.literal_eval(drugbank[drugbank['primary_drugbank_id']=="DB00003"].all_names.iloc[0])

['Deoxyribonuclease (human clone 18-1 protein moiety)',
 'Dornasa alfa',
 'Dornase alfa',
 'Dornase alfa, recombinant',
 'Dornase alpha',
 'Pulmozyme',
 'Recombinant deoxyribonuclease (DNAse)',
 'Viscozyme']

In [43]:
full_db[full_db['cui']=="C5544267"]

,cui,ispref,sab,str,scui,sty
2444166,C5544267,Y,MSH,SRP-4045,M000750391,"Nucleic Acid, Nucleoside, or Nucleotide"


In [19]:
current_db[current_db['cui']=="C0244672"]

,cui,srl,sab,str,sty
16031,C0244672,0,RXNORM,fluorodopa F-18,Organic Chemical


## Derive synonyms via UMLS and DrugBank

In [33]:
import ast
import pandas as pd

# 1) CUIs we care about
cuis = current_db["cui"].unique()

# 2) All UMLS rows for those CUIs
base = full_db[full_db["cui"].isin(cuis)].copy()

# base synonyms are just the UMLS strings
base_synonyms = base[["cui", "str"]].rename(columns={"str": "synonym"})

# 3) Subset to DRUGBANK source rows
db_rows = base[base["sab"] == "DRUGBANK"].copy()

# 4) Merge with DrugBank dataframe on ID
db_rows = db_rows.merge(
    drugbank[["primary_drugbank_id", "all_names"]],
    left_on="scui",
    right_on="primary_drugbank_id",
    how="left",
)

# 5) Ensure all_names is a list for each row
def to_list(val):
    if isinstance(val, list):
        return val
    if pd.isna(val):
        return []
    if isinstance(val, str):
        # try to parse "['name1', 'name2']"
        s = val.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, list):
                    return parsed
            except Exception:
                pass
        # otherwise treat as single string
        return [val]
    return []

db_rows["all_names_list"] = db_rows["all_names"].apply(to_list)

# 6) Explode one synonym per row
drugbank_synonyms = db_rows[["cui", "all_names_list"]].explode("all_names_list")
drugbank_synonyms = drugbank_synonyms.rename(columns={"all_names_list": "synonym"})

# 7) Concatenate UMLS synonyms + DrugBank synonyms
synonyms_df = pd.concat(
    [base_synonyms, drugbank_synonyms],
    ignore_index=True
)

# 8) Clean up synonyms
synonyms_df = synonyms_df.dropna(subset=["synonym"])
synonyms_df["synonym"] = synonyms_df["synonym"].astype(str).str.strip()
synonyms_df = synonyms_df[synonyms_df["synonym"] != ""]
synonyms_df = synonyms_df.drop_duplicates(subset=["cui", "synonym"])

# 9) EXCLUDE strings already present in current_db["str"]
existing_labels = (
    current_db["str"]
    .astype(str)
    .str.strip()
    .unique()
)

synonyms_df = synonyms_df[~synonyms_df["synonym"].isin(existing_labels)]
synonyms_df = synonyms_df.reset_index(drop=True)
synonyms_df = synonyms_df.rename(columns={"synonym": "str"})

synonyms_df["str"] = (
    synonyms_df["str"]
    .astype(str)      # ensure string
    .str.strip()      # remove surrounding spaces
    .str.lower()      # lowercase
)

# remove duplicates by cui + str
synonyms_df = synonyms_df.drop_duplicates(subset=["cui", "str"]).reset_index(drop=True)


In [70]:
db_rows.primary_drugbank_id.nunique()

10249

In [80]:
db_rows[['primary_drugbank_id']].drop_duplicates().to_csv("./data/term_dictionaries/drugbank_full_20251115/drugbank_from_umls.csv", index=False)


In [66]:
db_rows

,cui,ispref,sab,str,scui,sty,primary_drugbank_id,all_names,all_names_list
0,C5705059,Y,DRUGBANK,ASN-561,DB16988,Organic Chemical,DB16988,"['ASN-561', 'ASN120290']","[ASN-561, ASN120290]"
1,C5705064,Y,DRUGBANK,AF-710B,DB16911,Organic Chemical,DB16911,"['1-(2,8-Dimethyl-1-Thia-3,8-Diazaspiro[4.5]De...","[1-(2,8-Dimethyl-1-Thia-3,8-Diazaspiro[4.5]Dec..."
2,C5705066,Y,DRUGBANK,S-Apomorphine,DB16927,Organic Chemical,DB16927,"['(+)-10,11-Dihydroxyaporphine', '(6as)-5,6,6a...","[(+)-10,11-Dihydroxyaporphine, (6as)-5,6,6a,7-..."
3,C5705066,Y,DRUGBANK,"Apomorphine, s-",DB16927,Organic Chemical,DB16927,"['(+)-10,11-Dihydroxyaporphine', '(6as)-5,6,6a...","[(+)-10,11-Dihydroxyaporphine, (6as)-5,6,6a,7-..."
4,C5705066,Y,DRUGBANK,S-(+)-apomorphine,DB16927,Organic Chemical,DB16927,"['(+)-10,11-Dihydroxyaporphine', '(6as)-5,6,6a...","[(+)-10,11-Dihydroxyaporphine, (6as)-5,6,6a,7-..."
...,...,...,...,...,...,...,...,...,...
35161,C5556538,Y,DRUGBANK,"1-benzyl-8-methyl-1,4,8-triazaspiro(4.5)decan-...",DB18817,Organic Chemical,DB18817,"['1,4,8-triazaspiro(4.5)decan-2-one, 8-methyl-...","[1,4,8-triazaspiro(4.5)decan-2-one, 8-methyl-1..."
35162,C5556538,Y,DRUGBANK,"1,4,8-triazaspiro(4.5)decan-2-one, 8-methyl-1-...",DB18817,Organic Chemical,DB18817,"['1,4,8-triazaspiro(4.5)decan-2-one, 8-methyl-...","[1,4,8-triazaspiro(4.5)decan-2-one, 8-methyl-1..."
35163,C5667108,Y,DRUGBANK,MRTX-1719,DB18016,Organic Chemical,DB18016,['MRTX-1719'],[MRTX-1719]
35164,C5667208,Y,DRUGBANK,Farletuzumab ecteribulin,DB18857,Immunologic Factor,DB18857,['Eribulin/farletuzumab antibody drug conjugat...,[Eribulin/farletuzumab antibody drug conjugate...


In [34]:
synonyms_df.cui.nunique()

242907

In [37]:
synonyms_df

,cui,str
0,C5687455,solution for infusion or injection
1,C5687455,conventional release solution for infusion and...
2,C5687455,conventional release solution for infusion or ...
3,C5687455,conventional release solution for infusion and...
4,C5687455,conventional release solution for infusion or ...
...,...,...
826153,C5448248,dehydroepiandrosterone enanthate
826154,C5544338,"1-piperazinecarboxamide, 4-methyl-n-((1s)-2-ox..."
826155,C5544338,4-methylpiperazine-1-carboxylic acid [1-(3-ben...
826156,C5544338,4-methylpiperazine-1-carboxylic acid(1-((3-ben...


In [55]:
synonyms_df.shape

(826158, 2)

In [57]:
synonyms_df = synonyms_df[synonyms_df["str"].apply(lambda x: isinstance(x, str) and x.strip() != "")]
synonyms_df = synonyms_df.reset_index(drop=True)

In [59]:
synonyms_df.shape

(826158, 2)

In [39]:
synonyms_df[synonyms_df['cui']=="C0393022"]

,cui,str
38532,C0393022,rituximab
38533,C0393022,rituximab-containing product
38534,C0393022,product containing rituximab (medicinal product)
38535,C0393022,rituximab (substance)
38536,C0393022,"cd20 antibody, rituximab"
38537,C0393022,rituximab cd20 antibody
568661,C0393022,blitzima
568662,C0393022,mabthera
568663,C0393022,ritemvia
568664,C0393022,rituxan sc


In [41]:
synonyms_df.to_csv("./data/umls/mrconso_filtered_db_and_sty_synonyms.csv", index=False)

## Drugbank External IDs

In [123]:
# read JSON Lines file
db_ext_ids = pd.read_json("../drugbank_external_ids.jsonl", lines=True)

# keep only rows where external_id exists AND corresponds to a DrugBank ID
# (DrugBank IDs usually look like "DB00014", but adjust the condition as needed)

db_ext_ids = db_ext_ids[db_ext_ids["type"].notna()]
db_ext_ids.head()

,drugbank_id,scraped_at,type,value
1,DB00019,2025-11-19 16:53:24,external_id,HSP-130
2,DB00019,2025-11-19 16:53:24,external_id,MYL-1401H
3,DB00019,2025-11-19 16:53:24,external_id,PF-06881894
4,DB00019,2025-11-19 16:53:24,external_id,SD-01
6,DB00017,2025-11-19 16:53:25,external_id,SMC-021


In [125]:
db_rows_sub = db_rows[['cui', 'primary_drugbank_id']].drop_duplicates(subset=['primary_drugbank_id'])
db_rows_sub.head()

,cui,primary_drugbank_id
0,C5705059,DB16988
1,C5705064,DB16911
2,C5705066,DB16927
8,C5705589,DB16936
12,C5705655,DB16978


In [127]:
db_ext_ids_cui = db_ext_ids.merge(db_rows_sub, left_on="drugbank_id", right_on="primary_drugbank_id", how="left")
db_ext_ids_cui_syn = db_ext_ids_cui[['cui','value']]
db_ext_ids_cui_syn = db_ext_ids_cui_syn.rename(columns={"value": "str"})
db_ext_ids_cui_syn.head()

,cui,str
0,C4735048,HSP-130
1,C4735048,MYL-1401H
2,C4735048,PF-06881894
3,C4735048,SD-01
4,C0073994,SMC-021


In [131]:
db_ext_ids_cui_syn.to_csv("./data/umls/mrconso_filtered_db_and_sty_drugbank_external_ids.csv", index=False)